# Hard Attention (硬注意力) 实现

硬注意力只选择部分位置进行关注，是一种离散选择机制。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

## 实现硬注意力类

In [ ]:
class HardAttention:
    def __init__(self, hidden_dim, sample_method='stochastic'):
        self.hidden_dim = hidden_dim
        self.sample_method = sample_method
        self.W = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b = np.zeros((hidden_dim,))
    
    def compute_attention_probs(self, query, keys):
        scores = np.dot(keys, query)
        probs = softmax(scores)
        return probs
    
    def sample_location(self, probs):
        if self.sample_method == 'stochastic':
            location = np.random.choice(len(probs), p=probs)
        else:
            location = np.argmax(probs)
        return location
    
    def forward(self, query, keys, values, return_probs=False):
        probs = self.compute_attention_probs(query, keys)
        location = self.sample_location(probs)
        output = values[location]
        
        if return_probs:
            return output, location, probs
        return output, location

## 测试随机采样硬注意力

In [ ]:
hidden_dim = 64
seq_len = 10

query = np.random.randn(hidden_dim)
keys = np.random.randn(seq_len, hidden_dim)
values = np.random.randn(seq_len, hidden_dim)

attention = HardAttention(hidden_dim, sample_method='stochastic')

# 多次采样观察随机性
locations = []
for i in range(100):
    _, location = attention.forward(query, keys, values)
    locations.append(location)

# 可视化采样分布
plt.figure(figsize=(10, 4))
plt.hist(locations, bins=seq_len, edgecolor='black')
plt.xlabel('位置索引')
plt.ylabel('采样次数')
plt.title('硬注意力随机采样分布（100次）')
plt.grid(True, alpha=0.3)
plt.show()

## 测试贪婪硬注意力

In [ ]:
attention_greedy = HardAttention(hidden_dim, sample_method='greedy')
output, location, probs = attention_greedy.forward(query, keys, values, return_probs=True)

print(f"选中位置: {location}")
print(f"该位置概率: {probs[location]:.4f}")

# 可视化概率分布和选中位置
plt.figure(figsize=(10, 4))
bars = plt.bar(range(seq_len), probs)
bars[location].set_color('red')
plt.xlabel('位置索引')
plt.ylabel('注意力概率')
plt.title('硬注意力概率分布（红色为选中位置）')
plt.grid(True, alpha=0.3)
plt.show()

## Top-K硬注意力

In [ ]:
class HardAttentionTopK:
    def __init__(self, hidden_dim, k=3):
        self.hidden_dim = hidden_dim
        self.k = k
    
    def forward(self, query, keys, values):
        scores = np.dot(keys, query)
        locations = np.argsort(scores)[-self.k:]
        output = np.mean(values[locations], axis=0)
        return output, locations

# 测试
attention_topk = HardAttentionTopK(hidden_dim, k=3)
output, locations = attention_topk.forward(query, keys, values)

print(f"选中的Top-3位置: {locations}")

# 可视化
scores = np.dot(keys, query)
colors = ['red' if i in locations else 'blue' for i in range(seq_len)]
plt.figure(figsize=(10, 4))
plt.bar(range(seq_len), scores, color=colors)
plt.xlabel('位置索引')
plt.ylabel('注意力得分')
plt.title('Top-K硬注意力（红色为选中位置）')
plt.grid(True, alpha=0.3)
plt.show()